# The Power of Memory: Giving Your Agent Context

Up to this point, you have built agents that can:

- reason,
- call tools,
- solve multi-step problems,
- and handle errors.

That is already powerful.

But there is a serious limitation:

👉 Your agent forgets everything after each turn.

Every time you call the model, it starts from zero.

In real applications, this is not acceptable.

A real assistant must remember:

- what the user said before,
- what it answered earlier,
- important details like names or preferences.

In this lesson, we will understand why memory is essential,
and how to give your agent context.

In [37]:
import os
from dotenv import load_dotenv
import logging

from typing import Annotated, TypedDict, List

import operator

from langchain_core.prompts import PromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, AIMessage, RemoveMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt import ToolNode

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

GPT_MODEL = "gpt-5-nano"
ANTHROPIC_MODEL = "claude-opus-4-6"
load_dotenv(override=True)
logging.basicConfig(level=logging.INFO)

## The Necessity of State: Why Stateless Agents Fail

A stateless agent processes each request independently.

It does not remember:
- previous messages,
- previous answers,
- or any user information.

Each call is isolated.

This design may work for simple factual questions.
It fails in conversations.

Imagine speaking with someone who has no short-term memory.

You say:

> "Hello, my name is Yatoute."

A few seconds later, you ask:

> "What is my name?"

If the person answers:

> "Sorry, I don’t have any personal information about you."

You would feel frustrated.

That is exactly how a stateless chatbot behaves.

Let’s simulate this behavior using `ChatOpenAI`.

We will:

1. Introduce ourselves.
2. Then ask the model to recall our name.
3. Observe the failure.

In [3]:
model = ChatOpenAI(model=GPT_MODEL, temperature=0)

# Step 1: Introduce yourself
response_1 = model.invoke(
    "Hello, my name is Yatoute. Nice to meet you."
)

print("Bot:", response_1.content)

Bot: Nice to meet you, Yatoute! How can I help today? I can chat about your interests, help with tasks (like writing, planning, coding, or studying), answer questions, or practice a language. Tell me what you’d like to do or what you’re curious about.


The model answers politely.

But notice something important:

We did not store this message anywhere.
There is no conversation history.

Now let’s ask the second question.

In [4]:
# Step 2: Ask for the name again
response_2 = model.invoke(
    "What is my name?"
)

print("Bot:", response_2.content)

Bot: I don’t know your name. I don’t have access to personal information unless you share it here. If you’d like, tell me your name or a nickname and I’ll use it in this chat. For example, you can say “Call me Alex.”


Why Does This Fail?

The model usually answers something like:

> "I’m sorry, I don’t have any personal information about you."

This happens because the second call only contains:

"What is my name?"

The model does not see the previous message.

There is no memory.
There is no state.
There is no context.

Each request is independent.

This is what we call a **stateless system**.

## Implementing Short-Term Memory with LangGraph’s State Management

The stateless chatbot failed for one simple reason:

👉 each model call was isolated.

To fix this, we must **manage conversation history explicitly**.

A stateful agent must do two things:

- keep a memory of previous messages,
- pass that memory back to the model at every new turn.

LangGraph is designed for this.
It `StateGraph` gives you a clean way to store and update a shared state across steps.

> **The Core Pattern**

Here is the practical recipe we will follow:

1) **Store messages in the state**  
   Our `AgentState` contains a list of messages.  
   We use `Annotated[..., operator.add]` so new messages are appended automatically.

2) **Call the model with the full history**  
   The node sends `state["messages"]` to the LLM.  
   This means the model sees the conversation context.

3) **Loop outside the graph**  
   After each user input, we run the graph once.  
   The graph returns an updated state (with the new AI message),  
   and we reuse this updated state for the next turn.

This is short-term memory: the agent remembers what happened earlier in the same conversation.

> 1) Define the Stateful Agent State

The state contains a single key: `messages`.

Each time the node returns `{"messages": [new_message]}`,
LangGraph will append it to the existing list.

In [9]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]

> 2) Define the Core Node

This node is the agent’s “brain”.
It calls the LLM using the entire conversation history.

In real systems, this is where you might also:
- add a system prompt,
- inject retrieved context,
- or call tools.

For now, we only call the model.

In [10]:
def call_model(state: AgentState):
    """Invoke the LLM with the entire message history."""
    logging.info("--- NODE: call_model ---")

    model = ChatOpenAI(model=GPT_MODEL, temperature=0)
    response = model.invoke(state["messages"])

    return {"messages": [response]}

> 3) Build and Compile the Graph

This graph is minimal:
- one node (`agent`)
- then `END`

It runs one model call per turn.

In [11]:
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.set_entry_point("agent")
workflow.add_edge("agent", END)

app = workflow.compile()

> Running a Stateful Conversation (Interactive Loop)

Now we build a simple conversation loop.

Important idea:
- the graph itself runs one turn,
- the Python `while` loop handles multiple turns,
- the updated state is reused each time.

Type `exit` to stop.

In [38]:
def start_new_conversation(app):
    state: AgentState = {"messages": []}

    logging.info("--- Starting Stateful Conversation ---")
    print("Agent: Hello! How can I help you? (Type 'exit' to end)")

    while True:
        user_input = input("> User: ").strip()

        if user_input.lower() == "exit":
            print("Agent: Goodbye!")
            break

        print(f"\n> user: {user_input}")

        state["messages"].append(HumanMessage(content=user_input))
        state = app.invoke(state)

        ai_response = state["messages"][-1].content
        print(f"\n< Agent: {ai_response}")

In [13]:
start_new_conversation(app)

INFO:root:--- Starting Stateful Conversation ---


Agent: Hello! How can I help you? (Type 'exit' to end)


INFO:root:--- NODE: call_model ---



> user: Hello, My name is Yatoute !


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



< Agent: Nice to meet you, Yatoute! How can I help today? I can help with things like:
- answering questions and explaining ideas
- drafting emails or messages
- brainstorming ideas or plans
- translating or language practice
- writing, coding, or math help
- planning trips or projects
- summarizing articles or documents

What would you like to start with? And if you prefer French (or another language), I can reply in that too.


INFO:root:--- NODE: call_model ---



> user: What is my name?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



< Agent: Your name is Yatoute.

Would you like me to call you Yatoute or a nickname? And how can I help you today? If you prefer French or another language, I can switch too.
Agent: Goodbye!


> **Why This Works: The Power of Context**

The success of this agent comes entirely from memory.

Let’s analyze what the model receives during the second turn
when the user asks: “What is my name?”

- **Case 1 — Stateless Agent (Failure)**

During the second turn, when the user asks:

> "What is my name?"

The LLM receives only:

```python
[HumanMessage(content='What is my name?')]
```

There is no previous message.
There is no stored memory.
There is no context.

From the model’s point of view, this is a brand new conversation.

So the only logical answer is:

"I don’t have that information."

- **Case 2 — Stateful Agent (Success)**

Now let’s examine what the LLM receives in the stateful version.

At the moment the user asks:

> "What is my name?"

The LLM input is:

```python
[
    HumanMessage(content="My name is Yatoute...."),
    AIMessage(content="Hello Yatoute! ....."),
    HumanMessage(content="What is my name?")
]
```

This time, the answer is inside the context.

The model does not need special intelligence.
It simply reads the previous message in the history.

The correct answer becomes possible because
the information exists inside the state.

> **Key idea**

LLMs do not "remember" by themselves.

They only process what is inside their input.

If you give them the full conversation history,
they behave like they remember.

If you give them a single isolated message,
they behave like they forgot everything.

Memory is not magic.
Memory is state management.

## Building a Conversational Agent That Remembers Past Interactions

In the previous section, we demonstrated why memory is essential.
Without memory, an agent cannot hold a real conversation.

Earlier in the course, we also learned how to give an agent tools:
- search tools,
- calculators,
- custom Python functions,
- external APIs.

We saw that tools allow the agent to:
- access real-time information,
- perform precise computations,
- and interact with the outside world.

Now it is time to combine these two capabilities.

In this section, we will build an agent that:

- keeps a multi-turn conversation using memory,
- uses tools when necessary,
- and stores the results of those tool calls inside its memory.

This is the architecture used by most practical AI assistants.

When a user asks a question that requires a tool:
1. The agent reasons.
2. It calls the appropriate tool.
3. It receives the result.
4. It stores that result in memory.
5. It continues the conversation with full context.


To achieve this, we will expand our agent's understanding of "memory". 

In our earlier examples, memory was mostly:
- `HumanMessage`
- `AIMessage`

Now we add a third message type:

- **`HumanMessage`**  
  Represents what the user says.

- **`AIMessage`**  
  Represents what the assistant says.  
  This can be a final answer **or** a decision to call tools (via `tool_calls`).

- **`ToolMessage`**  
  Represents the output of a tool.  
  This is the *observation* after the agent takes an action.

Once tool outputs are stored as `ToolMessage` objects, the agent can reuse them later.
That is how the agent “remembers” what it discovered.

### The Conversational Agent’s Graph

The graph has two main nodes:

1) **`agent` node**  
   Calls the LLM using the full message history.

2) **`action` node** (toolkit node)  
   Executes any tool calls requested by the model and produces `ToolMessage` observations.

Then we use a conditional edge to decide:

- if the model requested a tool → go to the toolkit node  
- otherwise → end the turn

Finally, after tools run, we loop back to the model so it can read the tool output
and produce the final answer.

This is exactly the Reason → Act → Observe loop:
- model reasons and requests tools,
- tools run and produce observations,
- model uses observations to answer.

In [3]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a given city."""
    logging.info(f"--- calling get_weather with arg: city={city} ---")

    if "lagos" in city.lower():
        return "It is currently 28°C and sunny in Lagos."
    else:
        return f"I don't have the weather for {city}."
    
search_tool = TavilySearch(max_results=2)
tools = [get_weather, search_tool]

The `ToolNode` is a ready-made LangGraph node that:
- reads tool calls from the last `AIMessage`,
- executes them,
- and appends tool results as `ToolMessage` objects.

This saves us from writing manual tool execution code.

In [25]:
tool_node = ToolNode(tools=tools, messages_key="messages")

def call_model(state: AgentState):
    """The agent node: call the LLM with the full message history."""
    
    logging.info("--- NODE: call_model ---")
    
    model = ChatOpenAI(model=GPT_MODEL, temperature=0).bind_tools(tools)
    response = model.invoke(state["messages"])
    return {"messages": [response]}

### Conditional Edge: decide between tools or end

After the agent node runs, we inspect the last message.

- If it is an `AIMessage` that contains tool calls → continue to the toolkit.
- Otherwise → end the graph execution for this turn.

In [26]:
def should_continue(state: AgentState) -> str:
    logging.info("--- EDGE: should_continue ---")
    last_message = state["messages"][-1]

    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        logging.info("Decision: CONTINUE to toolkit (tools needed)")
        return "continue"
    else:
        logging.info("Decision: END (no tools needed)")
        return "end"

### Build the graph

In [28]:
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

workflow.set_entry_point("agent")

workflow.add_conditional_edges(
    source="agent",
    path=should_continue,
    path_map={
        "continue": "action",
        "end": END,
    },
)

workflow.add_edge("action", "agent")

app = workflow.compile()

### Running the Conversation

Each user message is appended to `state["messages"]`.
Then we run one graph execution (`app.invoke(state)`).

Because the state is reused every time,
the assistant sees the full conversation history on every turn.

In [4]:
def start_new_conversation(app: CompiledStateGraph):
    
    state: AgentState = {"messages": []}

    logging.info("--- Starting Stateful Conversation ---")
    print("Agent: Hello! How can I help you? (Type 'exit' to end)")

    while True:
        user_input = input("> User: ").strip()

        if user_input.lower() == "exit":
            print("Agent: Goodbye!")
            break
        
        print(f"\n> user: {user_input}")
        
        # Add the user message to memory
        state["messages"].append(HumanMessage(content=user_input))

        # Run one agent turn (one graph execution)
        state = app.invoke(state)

        # The latest message is the AI response
        ai_response = state["messages"][-1].content
        print(f"\n< Agent: {ai_response}")

In [31]:
start_new_conversation(app)

INFO:root:--- Starting Stateful Conversation ---


Agent: Hello! How can I help you? (Type 'exit' to end)


INFO:root:--- NODE: call_model ---



> user: What is the weather in Lagos? Please use tool provided to you if you need them.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:Decision: CONTINUE to toolkit (tools needed)
INFO:root:--- calling get_weather with arg: city=Lagos ---
INFO:root:--- NODE: call_model ---
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:Decision: END (no tools needed)



< Agent: It’s 28°C and sunny in Lagos right now.

Would you like an hourly forecast or a 7-day outlook? I can pull more details (humidity, wind, chances of rain) if you’d like.


INFO:root:--- NODE: call_model ---



> user: And what temperature did you say it was?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:Decision: END (no tools needed)



< Agent: 28°C (currently).
Agent: Goodbye!


### Let’s Trace a Conversation

Below is the real execution trace of our agent.
We will analyze what happens step by step.

The goal is to clearly see:

- when the model reasons,
- when a tool is called,
- when the graph loops,
- and when memory is used instead of tools.

#### Turn 1 — The Agent Uses a Tool

Conversation begins:

Agent: Hello! How can I help you? (Type 'exit' to end)

User:
> What is the weather in Lagos? Please use tool provided to you if you need them.

- **Step 1 — The Model Reasons**

Log:

INFO: --- NODE: call_model ---

The agent node runs.
The model reads the conversation history (at this point, only the user message).

It decides that answering requires a tool.

- **Step 2 — The Conditional Edge Checks**

Log:

INFO: --- EDGE: should_continue ---
INFO: Decision: CONTINUE to toolkit (tools needed)

Because the last AIMessage contains `tool_calls`,
the graph routes execution to the tool node.

- **Step 3 — The Tool Executes**

Log:

INFO: --- calling get_weather with arg: city=Lagos ---

The tool `get_weather` runs.
It produces the observation:

"It is currently 28°C and sunny in Lagos."

- **Step 4 — The Graph Loops Back to the Model**

Log:

INFO: --- NODE: call_model ---
INFO: --- EDGE: should_continue ---
INFO: Decision: END (no tools needed)

Now the model sees:

- the original user question
- the ToolMessage containing the weather result

Since it has all necessary information,
it produces a final answer and no longer requests tools.

> Final assistant response:

It’s 28°C and sunny in Lagos right now...

#### Turn 2 — The Agent Uses Memory (No Tool)

User now asks:

> And what temperature did you say it was?

- **Step 1 — The Model Reads the Full History**

Log:

INFO: --- NODE: call_model ---
INFO: --- EDGE: should_continue ---
INFO: Decision: END (no tools needed)

This time:

- No tool is called.
- The agent does not go to the tool node.

Why?

Because the conversation history already contains:

"It is currently 28°C and sunny in Lagos."

The model simply reads its own previous answer.

> Final assistant response:

28°C (currently).

### What We Achieved

This agent can now:

- hold a coherent multi-turn conversation,
- use tools when necessary,
- store tool results as part of memory,
- reuse those results in later turns.

This is one of the most common assistant architectures:
memory + tool use + an agent loop.

In the next lesson section, we will explore long-term memory strategies:
**summarization** and **vector stores**.

## Strategies for Long-Term Memory: Vector Stores and Summarization

The short-term memory we implemented works well for a single conversation session.

However, it has a natural limitation:

👉 LLMs have a limited context window.

Each model can only process a maximum number of tokens.
If we keep adding every message to `state["messages"]`,
eventually we will exceed that limit.

Sending a very long conversation history at every turn:

- increases cost,
- increases latency,
- and can even break the system if the token limit is exceeded.

A truly intelligent system must be able to remember information
without always sending everything back to the model.

This is where **long-term memory strategies** become essential.

There are two main approaches:

1. **Summarization**
2. **Retrieval using Vector Stores**

### Strategy 1 — Summarization

Summarization is the most intuitive approach.

Instead of keeping a full transcript forever,
the agent periodically creates a concise summary
of the conversation so far.

Then:

- the summary is stored in the state,
- old messages are removed,
- future turns use the summary as context.

The idea is simple:
compress the past into a smaller representation.

> **Advantages of Summarization:**

- Reduces token usage.
- Keeps important context.
- Easy to implement.
- Fully compatible with LangGraph state management.

> **Limitations of Summarization:**

- Some details may be lost.
- Summaries depend on the quality of the LLM.
- Errors in summarization may propagate.
- It is still limited to structured compression (not semantic retrieval).

Summarization trades precision for compression.

#### Implementing Automatic Conversation Summarization

We will extend our `AgentState`.

Now the state contains:

- `messages` → recent conversation
- `summary` → compressed memory of older conversation

In [5]:
# ==============================
# STATE
# ==============================

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    summary: str

> Tool Node

We reuse the tools defined in the previous example.

In [56]:
@tool
def get_weather(city: str) -> str:
    """Return the current weather for a given city."""
    logging.info(f"--- calling get_weather with arg: city={city} ---")

    if "lagos" in city.lower():
        return "It is currently 28°C and sunny in Lagos."
    else:
        return f"I don't have the weather for {city}."
    
search_tool = TavilySearch(max_results=2)
tools = [get_weather, search_tool]

tool_node = ToolNode(tools=tools, messages_key="messages")

> Model Node with Memory Injection

Now the agent injects the summary (if it exists)
as a `SystemMessage` before calling the model.

This allows the LLM to use compressed memory
without loading the full transcript.

In [52]:
# ==============================
# MODEL BINDING
# ==============================

model = ChatOpenAI(
    model=GPT_MODEL,
    temperature=0
).bind_tools(tools)

# ==============================
# AGENT NODE
# ==============================

def call_model(state: AgentState):
    logging.info("--- NODE: agent ---")

    system_prompt = SystemMessage(
        content=(
            "You are a precise and reliable AI agent.\n"
            "Use tools when necessary.\n"
            "Be concise and factual."
        )
    )

    summary_message = None
    if state.get("summary"):
        summary_message = SystemMessage(
            content=f"Conversation memory summary:\n{state['summary']}"
        )

    model_input = [system_prompt]

    if summary_message:
        model_input.append(summary_message)

    model_input.extend(state["messages"])

    response = model.invoke(model_input)

    return {"messages": [response]}

> Tool Routing

The tool loop remains the same:
if tools are required → execute them.

In [53]:
# ==============================
# TOOL ROUTING
# ==============================

def should_continue(state: AgentState) -> str:
    logging.info("--- EDGE: should_continue ---")

    last_message = state["messages"][-1]

    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        logging.info("→ Tools needed")
        return "continue"
    else:
        logging.info("→ No tools needed")
        return "end"

> Memory Update Logic

We now add a second decision point:

After a turn finishes,
should we update long-term memory?

In this example:
If more than 5 human messages exist,
we trigger summarization.

In [54]:
# ==============================
# MEMORY CHECK
# ==============================

def should_update_memory(state: AgentState) -> str:
    logging.info("--- EDGE: should_update_memory ---")

    human_messages = [
        m for m in state["messages"]
        if isinstance(m, HumanMessage)
    ]

    if len(human_messages) > 5:
        logging.info("→ Update memory")
        return "update_memory"
    else:
        return "end"

def check_memory(state: AgentState):
    return {}

> Summarization Node

This node:

1. Takes the existing summary (if any)
2. Extends or creates a new summary
3. Deletes older messages
4. Keeps only the last few recent messages

This prevents the context window from growing indefinitely.

In [55]:
# ==============================
# SUMMARIZE NODE
# ==============================

def summarize_node(state: AgentState):
    logging.info("--- NODE: summarize_node ---")

    summary = state.get("summary", "")

    if summary:
        prompt = (
            f"Existing summary:\n{summary}\n\n"
            "Extend this summary using the new conversation messages above."
        )
    else:
        prompt = "Create a concise summary of the conversation above."

    model_input = state["messages"] + [HumanMessage(content=prompt)]
    response = model.invoke(model_input)

    # Keep only last 4 messages
    messages_to_delete = state["messages"][:-4]

    delete_ops = [
        RemoveMessage(id=m.id)
        for m in messages_to_delete
    ]

    return {
        "summary": response.content,
        "messages": delete_ops
    }

> Build the Full Graph with Memory Branch

In [30]:
# ==============================
# BUILD GRAPH
# ==============================

workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)
workflow.add_node("check_memory", check_memory)
workflow.add_node("update_memory", summarize_node)

workflow.set_entry_point("agent")

# Tool loop
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "action",
        "end": "check_memory"
    }
)

workflow.add_edge("action", "agent")

# Memory branch
workflow.add_conditional_edges(
    "check_memory",
    should_update_memory,
    {
        "update_memory": "update_memory",
        "end": END
    }
)

workflow.add_edge("update_memory", END)

app = workflow.compile()

> Start the Conversation

In [31]:
start_new_conversation(app)

INFO:root:--- Starting Stateful Conversation ---


Agent: Hello! How can I help you? (Type 'exit' to end)


INFO:root:--- NODE: agent ---



> user: Hello, what is the score of the last match of Barcelona?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:→ Tools needed
INFO:root:--- NODE: agent ---
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:→ No tools needed
INFO:root:--- EDGE: should_update_memory ---


...


< Agent: Love the energy! Your excitement is exactly what makes match days fun. Vamos Barça!

Want me to:
- Set up live score updates and goal alerts?
- Share a quick pre-match lineup and key matchups?
- Give minute-by-minute live updates during the game?
- Post-match recap and player ratings?

Tell me what you’d like and I’ll tailor it.


INFO:root:--- NODE: agent ---



> user: Please can you remember me what was the score of the last match of Barcelona?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:→ No tools needed
INFO:root:--- EDGE: should_update_memory ---



< Agent: The last Barcelona match was Barcelona 3-0 Levante in La Liga on Feb 22, 2026. Scorers: Marc Bernal, Frenkie de Jong, and Fermín López. Want the full match report, lineup, or post-match highlights?
Agent: Goodbye!


#### What We Achieved

This agent now:

- Uses tools when necessary.
- Keeps short-term memory.
- Periodically summarizes long conversations.
- Prevents token overflow.
- Maintains context across many turns.

This is a scalable memory pattern used in real systems.


In the next section, we will explore the second long-term memory strategy:

**Retrieval using Vector Stores**, which allows semantic memory instead of compressed summaries.

### Strategy 2 — Retrieval with Vector Stores (RAG as Memory)

Summarization compresses the past into a short text.

Retrieval takes a different approach:
it treats memory like a **searchable database**.

This strategy is often called **RAG (Retrieval-Augmented Generation)**.
For long-term memory, RAG is powerful because the agent does **not** send the full history to the LLM.
Instead, it sends only the **most relevant** past information.

The workflow has three steps:

1) **Store**  
   After important interactions, we save key facts (user preferences, tool outputs, notes).
   We convert each text into a vector called an **embedding**, then store it in a **vector store**.

2) **Retrieve**  
   Before answering a new query, we embed the query and search the vector store.
   We retrieve the most similar memories.

3) **Augment**  
   We add the retrieved memories into the LLM input (as context),
   so the model answers with the right information.

## Setup: FAISS + Ollama Embeddings

We will use:

- **FAISS**: a popular vector store (fast, simple, in-memory)
- **Ollama**: to generate embeddings locally with the model `nomic-embed-text`

Install FAISS:

```bash
pip install faiss-cpu
````

For Ollama setup, follow the steps in `00_before_start.md`:

* install Ollama
* pull the embedding model `nomic-embed-text`
* start the Ollama server (`ollama serve`) 

#### 1) Build a Small “Long-Term Memory” Vector Store

For the demo, we create a few memories about “Reginald”.
In a real system, these memories would be added over time.

In [ ]:
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

long_term_memories = [
    "Reginald's favorite color is blue.",
    "Reginald is allergic to peanuts.",
    "Reginald's first pet was a dog named Sparky.",
    "Reginald works as a software developer.",
]

vector_store = FAISS.from_texts(
    texts=long_term_memories,
    embedding=embeddings,
    docstore=InMemoryDocstore()
)

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


#### 2) Define a Retrieval Node

This node runs *before* the LLM.

It reads the latest user question,
retrieves the top-k most relevant memories,
and injects them into the conversation as extra context.

Important idea:
- we don’t dump the full database into the prompt,
- we only add the most relevant pieces.

In [68]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [69]:
def retrieve_context(state: AgentState, k: int = 2):
    logging.info("--- NODE: retrieve_context ---")

    if not state.get("messages"):
        logging.warning("No messages found in state. Skipping retrieval.")
        return {"messages": []}

    last_message = state["messages"][-1]
    query_text = last_message.content

    retrieved_docs = vector_store.similarity_search(query=query_text, k=k)

    if not retrieved_docs:
        logging.info("No relevant documents retrieved.")
        return {"messages": []}

    # Simple serialization: we show the retrieved text to the LLM
    serialized = "\n".join(f"- {doc.page_content}" for doc in retrieved_docs)

    system_message = SystemMessage(
        content=(
            "You are given memory snippets retrieved from long-term memory.\n"
            "Use them only if they are relevant.\n"
            "If the memory does not contain the answer, say you don't know."
        )
    )

    memory_message = HumanMessage(content=f"Retrieved memory:\n{serialized}")

    logging.info(f"Retrieved {len(retrieved_docs)} documents for query: {query_text}")
    return {"messages": [system_message, memory_message]}

#### 3) The Agent Node + Optional Tools

We keep the model call simple.

In [70]:
# Agent Node
def call_model(state: AgentState):
    logging.info("--- NODE: call_model ---")

    model = ChatOpenAI(model=GPT_MODEL, temperature=0).bind_tools(tools)
    response = model.invoke(state["messages"])
    return {"messages": [response]}


> We reuse the existing `tools` and `tool_node` that were defined in the previous section. So the Agent can decide to use them when memory is not enough.

In [71]:
def should_continue(state: AgentState) -> str:
    logging.info("--- EDGE: should_continue ---")

    last_message = state["messages"][-1]
    if isinstance(last_message, AIMessage) and getattr(last_message, "tool_calls", None):
        logging.info("→ Tools needed")
        return "continue"
    logging.info("→ No tools needed")
    return "end"

#### 4) Build the Graph

Flow:

START → retrieve_context → call_model → (tools?) → END

In [72]:
builder = StateGraph(AgentState)

builder.add_node("context", retrieve_context)
builder.add_node("agent", call_model)

builder.add_edge(START, "context")
builder.add_edge("context", "agent")

if tool_node is not None:
    builder.add_node("action", tool_node)

    builder.add_conditional_edges(
        source="agent",
        path=should_continue,
        path_map={"continue": "action", "end": END},
    )
    builder.add_edge("action", "agent")
else:
    builder.add_edge("agent", END)

app = builder.compile()

#### 5) Run the Conversation

In [66]:
start_new_conversation(app)

INFO:root:--- Starting Stateful Conversation ---


Agent: Hello! How can I help you? (Type 'exit' to end)


INFO:root:--- NODE: retrieve_context ---
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:root:Retrieved 2 documents for query: Who is Reginald?
INFO:root:--- NODE: call_model ---



> user: Who is Reginald?


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:→ No tools needed


...


< Agent: I don’t have any information about Cristiano Ronaldo in the provided context. The sources you gave only mention Reginald’s job and his favorite color. If you’d like, I can look up general information about Cristiano Ronaldo from reliable sources. Would you like me to search the web?


INFO:root:--- NODE: retrieve_context ---
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:root:Retrieved 2 documents for query: Yes use tool provided to search
INFO:root:--- NODE: call_model ---



> user: Yes use tool provided to search


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:→ Tools needed
INFO:root:--- NODE: call_model ---
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:--- EDGE: should_continue ---
INFO:root:→ No tools needed



< Agent: Cristiano Ronaldo is a Portuguese professional football forward widely regarded as one of the greatest players of his generation. He was born on February 5, 1985, in Funchal, Madeira, Portugal. He has won five Ballon d’Or awards (2008, 2013, 2014, 2016, 2017) and, as of 2024, became the first man to score 900 career goals in official matches.

References:
- Britannica: Cristiano Ronaldo | Biography, Football, & Facts
- Amazon: Cristiano Ronaldo: The Biography
Agent: Goodbye!


This architecture allows the agent to:

1. Use long-term memory first (vector store retrieval).
2. Avoid hallucination when memory is insufficient.
3. Escalate to external tools when needed.
4. Combine memory and tools in the same system.

The agent now has three layers of knowledge:

- Short-term conversation history
- Long-term vector memory
- External tools (search, APIs, etc.)

This is a scalable and production-ready memory pattern.

What Comes Next ?

We have now built:

- A stateful conversational agent
- With short-term memory
- With long-term vector memory
- And with tool integration

In the next section, we will move from theory to practice.

> **Project: Chat PDF**

An agent that reads a PDF file and answers questions about its content.

In this project, you will:

- Load a PDF document
- Chunk and embed its content
- Store it in a vector store
- Build a conversational RAG agent over the document

This will be your first complete memory-powered application.